<a href="https://colab.research.google.com/github/Valdes-Tresanco-MS/gmx_MMPBSA/blob/colab-notebook/notebooks/gmx_MMPBSA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# gmx_MMPBSA examples on Google Colab CPUs

This notebook installs a CPU-only gmx_MMPBSA environment in Google Colab, runs selected bundled examples, and displays the generated result files. It is intended as a quick runtime check and a starting point for adapting commands to your own systems.

Use **Runtime > Change runtime type > CPU** before running the notebook. A fresh Colab runtime is recommended.

## Runtime notes

gmx_MMPBSA currently requires Python 3.11. Colab runtime Python versions change over time, so this notebook creates an isolated conda environment in `/content/miniforge3` and runs gmx_MMPBSA commands through `conda run`. The notebook kernel can stay on Colab's default Python.

In [ ]:
import os
import platform
import os
import multiprocessing
from pathlib import Path

print("Python kernel:", platform.python_version())
print("Platform:", platform.platform())
print("CPU cores:", multiprocessing.cpu_count())
print("Working directory:", Path.cwd())

## 1. Install gmx_MMPBSA and dependencies

This cell installs Miniforge if needed, creates a `gmxMMPBSA` conda environment, installs AmberTools, GROMACS, MPI support, and installs gmx_MMPBSA from the same GitHub branch as this notebook. It can take several minutes on a fresh Colab runtime.

In [ ]:
import os
import select
import subprocess
import time
from datetime import datetime
from pathlib import Path

MINIFORGE = Path("/content/miniforge3")
CONDA = MINIFORGE / "bin" / "conda"

def log(message):
    print(f"\n[{datetime.now():%H:%M:%S}] {message}", flush=True)

def run_step(label, cmd, heartbeat_seconds=20):
    log(label)
    print("$ " + " ".join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        [str(part) for part in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    os.set_blocking(process.stdout.fileno(), False)
    last_output = time.monotonic()
    while process.poll() is None:
        ready, _, _ = select.select([process.stdout], [], [], 1)
        if ready:
            chunk = os.read(process.stdout.fileno(), 4096)
            if chunk:
                print(chunk.decode(errors="replace"), end="", flush=True)
                last_output = time.monotonic()
        elif time.monotonic() - last_output >= heartbeat_seconds:
            print(f"[{datetime.now():%H:%M:%S}] Still working on: {label}", flush=True)
            last_output = time.monotonic()
    remaining = process.stdout.read()
    if remaining:
        print(remaining.decode(errors="replace"), end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, cmd)

log("Starting gmx_MMPBSA Colab environment setup")
print(f"Install prefix: {MINIFORGE}", flush=True)

if not CONDA.exists():
    run_step(
        "Downloading Miniforge installer",
        ["wget", "--progress=dot:giga", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", "/tmp/miniforge.sh"],
    )
    run_step("Installing Miniforge", ["bash", "/tmp/miniforge.sh", "-b", "-p", str(MINIFORGE)])
else:
    log("Reusing existing Miniforge installation")

run_step("Configuring conda channel priority", [CONDA, "config", "--set", "channel_priority", "strict"])

env_list = subprocess.run([CONDA, "env", "list"], text=True, stdout=subprocess.PIPE, check=True).stdout
env_exists = any(line.split() and line.split()[0] == "gmxMMPBSA" for line in env_list.splitlines())
if not env_exists:
    print("\nThis is the longest step. Conda is solving and installing AmberTools, GROMACS, MPI, and Python packages.", flush=True)
    run_step(
        "Creating gmxMMPBSA environment",
        [
            CONDA, "create", "-n", "gmxMMPBSA", "-c", "conda-forge", "-y",
            "python=3.11", "pip", "git", "ambertools<24", "gromacs<2026",
            "mpi4py=4.0.1", "numpy=1.26.4", "scipy=1.14.1",
            "pandas=1.5.3", "matplotlib=3.7.3", "seaborn=0.11.2",
            "parmed>=4.2.2", "tqdm",
        ],
    )
else:
    log("Reusing existing gmxMMPBSA conda environment")

GITHUB_BRANCH = "colab-notebook"
GITHUB_INSTALL = f"git+https://github.com/Valdes-Tresanco-MS/gmx_MMPBSA.git@{GITHUB_BRANCH}"
run_step("Installing or updating gmx_MMPBSA from GitHub", [CONDA, "run", "-n", "gmxMMPBSA", "python", "-m", "pip", "install", "--no-deps", "-U", GITHUB_INSTALL])
run_step("Installed command versions", [CONDA, "run", "-n", "gmxMMPBSA", "bash", "-lc", "python --version && gmx_MMPBSA --version && gmx --version | sed -n '1,4p'"])
run_step("Cleaning conda package cache", [CONDA, "clean", "-afy"])

log("Environment setup finished")

## 2. Verify installed tools

In [ ]:
%%bash
set -euo pipefail
CONDA=/content/miniforge3/bin/conda
ENV_PREFIX=/content/miniforge3/envs/gmxMMPBSA

"$CONDA" run -n gmxMMPBSA python --version
"$CONDA" run -n gmxMMPBSA gmx_MMPBSA --version

if [ -x "$ENV_PREFIX/bin/amber_MMPBSA" ]; then
  "$ENV_PREFIX/bin/amber_MMPBSA" --version
else
  echo "amber_MMPBSA command not found; skipping AMBER-only example test 25."
fi

"$CONDA" run -n gmxMMPBSA gmx --version | sed -n '1,8p'
"$CONDA" run -n gmxMMPBSA mpirun --version | sed -n '1,4p'

## 3. Run bundled examples

The default runs the protein-ligand single-trajectory example (`-t 3`) with 1 MPI rank. You can change `TEST_SELECTION`, `NUM_PROCESSORS`, and `NUM_CONCURRENT` before running the next cell. Some Colab runtimes expose fewer OpenMPI slots than Python CPU threads, so the run cell probes MPI first and falls back to one rank if needed. The notebook installs the current GitHub branch, but the run cell still skips `-j` defensively if an older environment is reused.

Avoid setting `NUM_PROCESSORS * NUM_CONCURRENT` higher than the CPU count reported above.

In [ ]:
# gmx_MMPBSA_test selections:
# 3 = Protein-Ligand (Single trajectory approximation)
# 2 = Fast set, 1 = Minimal set, 0 = All examples
# 25 = AMBER input files in newer versions; use only if listed in gmx_MMPBSA_test -h and amber_MMPBSA is available
TEST_SELECTION = "3"
NUM_PROCESSORS = 1
NUM_CONCURRENT = 1
WORKDIR = "/content"

In [ ]:
import os
import multiprocessing
import shlex
import subprocess

cpu_count = multiprocessing.cpu_count()
env = os.environ.copy()
env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
env["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

effective_processors = int(NUM_PROCESSORS)
requested = effective_processors * int(NUM_CONCURRENT)
if requested > cpu_count:
    print(f"Warning: requested up to {requested} MPI ranks on {cpu_count} CPU cores.")

conda_cmd = ["/content/miniforge3/bin/conda", "run", "-n", "gmxMMPBSA"]
help_result = subprocess.run(
    conda_cmd + ["gmx_MMPBSA_test", "-h"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
help_text = help_result.stdout
supports_concurrency = "-j" in help_text or "--num_concurrent" in help_text

if effective_processors > 1:
    mpi_probe = subprocess.run(
        conda_cmd + ["mpirun", "-np", str(effective_processors), "true"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    if mpi_probe.returncode:
        print(f"OpenMPI rejected -np {effective_processors} in this Colab runtime; using -n 1 instead.")
        print(mpi_probe.stdout.splitlines()[0] if mpi_probe.stdout else "No MPI probe output.")
        effective_processors = 1


cmd = conda_cmd + [
    "gmx_MMPBSA_test", "-f", WORKDIR, "-ng",
    "-n", str(effective_processors),
    "-t", *shlex.split(str(TEST_SELECTION)),
]
if supports_concurrency:
    cmd.extend(["-j", str(NUM_CONCURRENT)])
elif int(NUM_CONCURRENT) != 1:
    print("This gmx_MMPBSA_test version does not support -j/--num_concurrent; running examples sequentially.")

print("Running:", " ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
print(result.stdout)
if result.returncode:
    raise RuntimeError(f"gmx_MMPBSA_test failed with exit code {result.returncode}")

## 4. Inspect result files

In [ ]:
from pathlib import Path

base = Path(WORKDIR) / "gmx_MMPBSA_test" / "examples"
all_mmxsa_files = sorted(base.rglob("COMPACT_MMXSA_RESULTS.mmxsa"))
mmxsa_files = [path for path in all_mmxsa_files if "API" not in path.relative_to(base).parts]
api_sample_files = [path for path in all_mmxsa_files if "API" in path.relative_to(base).parts]
result_files = sorted(base.rglob("FINAL_RESULTS_MMPBSA.dat"))
csv_files = sorted(base.rglob("FINAL_RESULTS_MMPBSA.csv"))
log_files = sorted(base.rglob("*.log"))

print("Compact API result files from calculations:")
for path in mmxsa_files:
    print(" -", path)
if not mmxsa_files and api_sample_files:
    print("No calculation compact result found. API sample files are available:")
    for path in api_sample_files:
        print(" -", path)

print("Result text files:")
for path in result_files:
    print(" -", path)

print("\nResult CSV files:")
for path in csv_files:
    print(" -", path)

print("\nLog files:")
for path in log_files:
    print(" -", path)

In [ ]:
if result_files:
    print(result_files[0])
    print(result_files[0].read_text(errors="replace")[:8000])
else:
    print("No FINAL_RESULTS_MMPBSA.dat files found.")

In [ ]:
import subprocess
import textwrap

if mmxsa_files:
    result_path = mmxsa_files[0]
    print(result_path)
    code = f"""
from pathlib import Path
from GMXMMPBSA import API

result_file = Path({str(result_path)!r})
if hasattr(API, 'load'):
    api = API.load(result_file)
else:
    api = API.MMPBSA_API()
    api.load_file(result_file)
info = api.get_info()
energy = api.get_energy()

print('Frames:', info.get('numframes'))
print('Energy map:', energy['map'])

gb_delta = energy['data']['normal']['gb']['delta']
print('GB delta energy table:')
print(gb_delta.head().to_string())
print('GB delta summary:')
print(energy['summary']['normal']['gb']['delta'].to_string())
"""
    completed = subprocess.run(
        ["/content/miniforge3/bin/conda", "run", "-n", "gmxMMPBSA", "python", "-c", textwrap.dedent(code)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode:
        raise RuntimeError(f"API result loading failed with exit code {completed.returncode}")
elif csv_files:
    print("No COMPACT_MMXSA_RESULTS.mmxsa file found; showing the raw CSV path instead:")
    print(csv_files[0])
else:
    print("No API result or FINAL_RESULTS_MMPBSA.csv files found.")

## Next step: adapt to your own files

After this example run works, use the command printed by each example README as the template for your own data. In a future upload-oriented version, this notebook can add file upload or Google Drive mounting cells and build the `gmx_MMPBSA` command from user-provided file paths.